In [10]:
import os
import json
import chromadb
import voyageai
import anthropic
import firebase_admin
from firebase_admin import credentials, firestore

# Service account path
SERVICE_ACCOUNT_PATH = "../credentials/our-kitchen-app-5557e-firebase-adminsdk-fbsvc-7b99d2edb7.json"

# Firebase
if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
    firebase_admin.initialize_app(cred)
db = firestore.client()

# Load secrets first
with open("/Users/sagar/dev/TheBrain/secrets.json") as f:
    secrets = json.load(f)

# Then initialize clients with keys
client = anthropic.Anthropic(api_key=secrets["ANTHROPIC_API_KEY"])
voyage_client = voyageai.Client(api_key=secrets["VOYAGE_API_KEY"])

# Reconnect to ChromaDB
chroma_client = chromadb.PersistentClient(path="/Users/sagar/dev/TheBrain/notebooks/ourbrain_chroma")
collection = chroma_client.get_collection("meals")

print(f"Connected. Meals in index: {collection.count()}")

Connected. Meals in index: 11


In [4]:
def semantic_search(query, n_results=3):
    # Step 1: convert the query text into a vector using the same
    # Voyage AI model we used to embed our meals. This is critical —
    # query and meals must live in the same vector space to be comparable.
    query_vector = voyage_client.embed([query], model="voyage-2").embeddings[0]
    
    # Step 2: ask ChromaDB to find the n most similar meal vectors
    # to our query vector. It measures similarity using cosine distance —
    # lower distance means more semantically similar.
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results
    )
    
    # Step 3: reformat the raw ChromaDB results into a clean list
    # of dicts that are easier to work with in the rest of the notebook.
    # ChromaDB returns parallel lists (documents, metadatas, distances)
    # so we zip them together into one object per meal.
    meals = []
    for i in range(len(results["documents"][0])):
        meals.append({
            "name": results["metadatas"][0][i].get("name"),
            "distance": round(results["distances"][0][i], 4),
            "chunk": results["documents"][0][i]
        })
    
    return meals

def print_results(query, results):
    # Simple display helper — formats results consistently
    # across all the test queries in cells 3, 4, and 5.
    print(f"Query: '{query}'")
    print("-" * 50)
    for r in results:
        print(f"  {r['name']} (distance: {r['distance']})")
    print()

In [5]:
# These queries map to real Firestore fields
# RAG should still return good results but so would a tool call
structured_queries = [
    "high protein meals",
    "chicken dishes",
    "Asian cuisine",
    "Italian food",
]

for q in structured_queries:
    results = semantic_search(q)
    print_results(q, results)

Query: 'high protein meals'
--------------------------------------------------
  Bahn Mi Subs (distance: 0.2479)
  Bibimbap (distance: 0.2551)
  Chicken Alfredo with Broccoli (distance: 0.268)

Query: 'chicken dishes'
--------------------------------------------------
  Chicken Tikka Masala (distance: 0.1989)
  Chicken quesadillas (distance: 0.2078)
  Pesto chicken pasta (distance: 0.2234)

Query: 'Asian cuisine'
--------------------------------------------------
  Bibimbap (distance: 0.1941)
  Bahn Mi Subs (distance: 0.227)
  Sushi Bake with Salmon and Shrimp (distance: 0.233)

Query: 'Italian food'
--------------------------------------------------
  Pesto chicken pasta (distance: 0.2278)
  Chicken Alfredo with Broccoli (distance: 0.2391)
  Ground Beef Shawarma (distance: 0.2627)



In [6]:
# These queries have no matching Firestore field
# Only semantic search can answer these
fuzzy_queries = [
    "something light and fresh",
    "comfort food for a cold night",
    "meals that are good for meal prep",
    "quick and easy weeknight dinner",
    "something filling but not too heavy",
]

for q in fuzzy_queries:
    results = semantic_search(q)
    print_results(q, results)

Query: 'something light and fresh'
--------------------------------------------------
  Bibimbap (distance: 0.3126)
  Salmon and Roasted Veggies (distance: 0.3136)
  Bahn Mi Subs (distance: 0.3193)

Query: 'comfort food for a cold night'
--------------------------------------------------
  Bahn Mi Subs (distance: 0.2631)
  Bibimbap (distance: 0.2659)
  Chicken Tikka Masala (distance: 0.2768)

Query: 'meals that are good for meal prep'
--------------------------------------------------
  Bahn Mi Subs (distance: 0.2501)
  Bibimbap (distance: 0.2643)
  Sushi Bake with Salmon and Shrimp (distance: 0.2854)

Query: 'quick and easy weeknight dinner'
--------------------------------------------------
  Bahn Mi Subs (distance: 0.2387)
  Bibimbap (distance: 0.2542)
  Chicken Tikka Masala (distance: 0.2543)

Query: 'something filling but not too heavy'
--------------------------------------------------
  Bahn Mi Subs (distance: 0.3056)
  Bibimbap (distance: 0.3096)
  Ground Beef Shawarma (distanc

In [7]:
# These should ideally return low confidence or irrelevant results
# They test whether your index knows what it doesn't know
edge_queries = [
    "desserts and sweets",        # probably nothing relevant
    "breakfast foods",            # probably nothing relevant  
    "vegan meals",                # depends on your data
    "something we haven't made",  # nonsense query
]

for q in edge_queries:
    results = semantic_search(q)
    print_results(q, results)

Query: 'desserts and sweets'
--------------------------------------------------
  Greek yogurt chocolate muffins (distance: 0.2882)
  Sushi Bake with Salmon and Shrimp (distance: 0.3142)
  Bahn Mi Subs (distance: 0.3172)

Query: 'breakfast foods'
--------------------------------------------------
  Greek yogurt chocolate muffins (distance: 0.2369)
  Greek Yogurt Pancakes (distance: 0.255)
  Bahn Mi Subs (distance: 0.256)

Query: 'vegan meals'
--------------------------------------------------
  Bahn Mi Subs (distance: 0.2272)
  Bibimbap (distance: 0.2305)
  Chicken Tikka Masala (distance: 0.2537)

Query: 'something we haven't made'
--------------------------------------------------
  Bahn Mi Subs (distance: 0.3283)
  Sushi Bake with Salmon and Shrimp (distance: 0.332)
  Bibimbap (distance: 0.3328)



In [11]:
# This is the full Phase 3c pattern — retrieval feeding into generation
def ask_ourbrain(question):
    # Step 1: retrieve relevant meals semantically
    retrieved = semantic_search(question, n_results=3)
    
    # Step 2: format retrieved meals as context
    context = "\n".join([r["chunk"] for r in retrieved])
    
    # Step 3: pass context + question to Claude
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        system="""You are OurBrain, a personal household AI assistant. 
You have access to the household's meal history. 
Answer questions based only on the meal data provided. 
Be conversational and specific.""",
        messages=[{
            "role": "user",
            "content": f"""Here are the most relevant meals from our history:

{context}

Question: {question}"""
        }]
    )
    
    return response.content[0].text

# Test it
questions = [
    "What are some high protein meals we've made?",
    "What's a good light meal we've cooked before?",
    "Suggest an Asian meal we've made for a weeknight dinner",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_ourbrain(q)}")
    print("=" * 60)

Q: What are some high protein meals we've made?
A: Based on your meal history, here are some **high protein meals** you've made:

1. **Bánh Mì Subs** 🥖 - Packed with **chicken breast** as the main protein source, plus the egg-based mayonnaise adds a bit more.

2. **Bibimbap** 🍚 - Great protein from **beef bulgogi**, plus an **egg** on top. The bean sprouts and spinach also contribute some plant-based protein.

3. **Chicken Quesadillas** 🫓 - A solid protein combo with **chicken breast** and a **Mexican cheese blend**.

All three meals actually feature a solid animal protein as a centerpiece! The **Bibimbap** might edge out the others since it has both beef *and* eggs. The **Bánh Mì and Quesadillas** both lean on chicken breast, which is a great lean protein option.

Would you like to revisit any of these meals soon, or are you looking for something new along these lines? 😊
Q: What's a good light meal we've cooked before?
A: Based on your meal history, **Salmon and Roasted Veggies** woul